### Introduction to Data Ingestion

In [1]:
import langchain
from typing import List, Dict, Any # tells which data types/parameters we used
import pandas as pd

In [8]:
from langchain_core.documents import Document # For storing document content and metadata

print("Set up completed !")

Set up completed !


### Understanding Document Structure in Langchain

In [3]:
# Create a simple document

doc = Document(
    page_content = "this is the main text which will be embedded",
    metadata = {
        "source":"example.txt",
        "page": 1,
        "author":"Rimu Roy",

    }
)
print("Document structure .")
print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

# why metadata mattters:
 #   1. Filtering search results
  #  2. Tracking documents sources
   # 3. Debugging 

Document structure .
Content: this is the main text which will be embedded
Metadata: {'source': 'example.txt', 'page': 1, 'author': 'Rimu Roy'}


### Text files (.txt) The simplest Case

In [4]:
### Create a simple text files
import os
os.makedirs("data/text_files",exist_ok=True)     # Create directories

In [5]:
sample_texts = {
    "data/text_files/python_intro.txt":"""Introduction to Python
Python is a high-level, interpreted programming language known for its simplicity, readability, and versatility.
Created by Guido van Rossum and first released in 1991, Python emphasizes code readability through its clean syntax
and use of significant whitespace. It supports multiple programming paradigms, including procedural, object-oriented, 
and functional programming, making it suitable for a wide range of applications from web development and
 data analysis to artificial intelligence and scientific computing. Python's extensive standard library and 
 vast ecosystem of third-party packages (like NumPy, Pandas, and TensorFlow) have made it one of the most 
 popular programming languages in the world, particularly in fields like data science, machine learning, and
   automation. Its interpreted nature allows for rapid development and testing, while its strong community
support ensures continuous improvement and abundant learning resources. Whether you're a beginner taking your
 first steps in coding or an experienced developer building complex systems, Python's "batteries-included" 
 philosophy and gentle learning curve make it an excellent choice for almost any programming task.

Key Points:

Created: 1991 by Guido van Rossum

Paradigm: Multi-paradigm (OOP, procedural, functional)

Key Features: Readable syntax, extensive libraries, interpreted

Popular Uses: Web development, Data Science, AI/ML, Automation

Community: Large, active, beginner-friendly



""",
    "data/text_files/machine_learning_intro.txt":"""Introduction to machine learning
Machine Learning (ML) is a transformative subset of artificial intelligence that enables computer systems to learn from data and improve their performance over time without being explicitly programmed. Instead of following rigid, pre-defined rules, ML algorithms identify patterns and make decisions by analyzing vast amounts of data, much like humans learn from experience. At its core, machine learning involves feeding data into algorithms that build mathematical models, which are then used to make predictions or decisions. This field is broadly categorized into three main types: Supervised Learning (learning from labeled examples, like email spam detection), Unsupervised Learning (finding hidden patterns in unlabeled data, like customer segmentation), and Reinforcement Learning (learning through trial and error, like game-playing AI). Machine learning is already deeply integrated into our daily lives, powering everything from personalized streaming recommendations and voice assistants to autonomous vehicles, medical diagnostics, and fraud detection in banking. With its ability to automate complex decision-making and uncover insights from massive datasets, machine learning has become a cornerstone of modern technology, driving innovation across industries and reshaping how we interact with the digital world.

Key Points:

Definition: Systems that learn from data without explicit programming

Main Types: Supervised, Unsupervised, Reinforcement Learning

Popular Algorithms: Linear Regression, Decision Trees, Neural Networks, SVM

Real-World Uses: Recommendations (Netflix/Amazon), Voice Assistants, Self-driving Cars, Medical Diagnosis, Spam Filters

Core Concept: "Data + Algorithms = Models that improve with more data"
"""
} 
for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f: # encoding = "utf-8" -> save using universal text format
        f.write(content)

print("Sample text files created")

Sample text files created


### TextLoader - Read single file

In [6]:
from langchain_community.document_loaders import TextLoader  # import textloader to load text file from a .txt file

loader = TextLoader("data/text_files/python_intro.txt",encoding="utf-8") # to read the text file
documents = loader.load()   # Load the entire text file
print(type(documents))
print(f"Loaded {len(documents)} documents")
print(f"Content preview: {documents[0].page_content[:100]}")
print(f"Metadata: {documents[0].metadata}")

/tmp/ipykernel_862/598219376.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader  # import textloader to load text file from a .txt file


<class 'list'>
Loaded 1 documents
Content preview: Introduction to Python
Python is a high-level, interpreted programming language known for its simpli
Metadata: {'source': 'data/text_files/python_intro.txt'}


### DirectoryLoader - Multiple text files

In [7]:
from langchain_community.document_loaders import DirectoryLoader

# Load all the text files from the directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob = "**/*.txt",   # Pattern to match files
    loader_cls= TextLoader, # Loader class to use
    loader_kwargs= {'encoding':'utf-8'},
    show_progress= True
)
documents = dir_loader.load()

for i,doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"Source: {doc.metadata['source']}")
    print(f"Length: {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 321.65it/s]


Document 1:
Source: data/text_files/machine_learning_intro.txt
Length: 1778 characters

Document 2:
Source: data/text_files/python_intro.txt
Length: 1506 characters


### Text splitting strategies

In [9]:
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print(documents)

[Document(metadata={'source': 'data/text_files/machine_learning_intro.txt'}, page_content='Introduction to machine learning\nMachine Learning (ML) is a transformative subset of artificial intelligence that enables computer systems to learn from data and improve their performance over time without being explicitly programmed. Instead of following rigid, pre-defined rules, ML algorithms identify patterns and make decisions by analyzing vast amounts of data, much like humans learn from experience. At its core, machine learning involves feeding data into algorithms that build mathematical models, which are then used to make predictions or decisions. This field is broadly categorized into three main types: Supervised Learning (learning from labeled examples, like email spam detection), Unsupervised Learning (finding hidden patterns in unlabeled data, like customer segmentation), and Reinforcement Learning (learning through trial and error, like game-playing AI). Machine learning is already 

In [10]:
 text = documents[0].page_content
 text

'Introduction to machine learning\nMachine Learning (ML) is a transformative subset of artificial intelligence that enables computer systems to learn from data and improve their performance over time without being explicitly programmed. Instead of following rigid, pre-defined rules, ML algorithms identify patterns and make decisions by analyzing vast amounts of data, much like humans learn from experience. At its core, machine learning involves feeding data into algorithms that build mathematical models, which are then used to make predictions or decisions. This field is broadly categorized into three main types: Supervised Learning (learning from labeled examples, like email spam detection), Unsupervised Learning (finding hidden patterns in unlabeled data, like customer segmentation), and Reinforcement Learning (learning through trial and error, like game-playing AI). Machine learning is already deeply integrated into our daily lives, powering everything from personalized streaming re

In [11]:
# Method 1
print("Character Text Splitter")
char_splitter = CharacterTextSplitter(
  separator="\n",  # split on newlines
  chunk_size = 200, # Max chunk size
  chunk_overlap = 20, # Overlap between chunks
  length_function = len # how to measure chunk size 
)
char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

Created a chunk of size 1328, which is longer than the specified 200


Character Text Splitter
Created 5 chunks
First chunk: Introduction to machine learning...


In [13]:
print(char_chunks[0])
print("-----")
print(char_chunks[1])
print("-----")
print(char_chunks[2])
print("-----")
print(char_chunks[3])
print("-----")

Introduction to machine learning
-----
Machine Learning (ML) is a transformative subset of artificial intelligence that enables computer systems to learn from data and improve their performance over time without being explicitly programmed. Instead of following rigid, pre-defined rules, ML algorithms identify patterns and make decisions by analyzing vast amounts of data, much like humans learn from experience. At its core, machine learning involves feeding data into algorithms that build mathematical models, which are then used to make predictions or decisions. This field is broadly categorized into three main types: Supervised Learning (learning from labeled examples, like email spam detection), Unsupervised Learning (finding hidden patterns in unlabeled data, like customer segmentation), and Reinforcement Learning (learning through trial and error, like game-playing AI). Machine learning is already deeply integrated into our daily lives, powering everything from personalized streamin